# 手持运动模拟扫描（双轴并行）

使用二轴机械平台模拟手持扫描：
- **X 轴**：匀速运动（与机械扫描相同）
- **Y 轴**：在 X 扫描期间并行执行一系列 `jog_single_axis_relative` 小幅相对运动，模拟手持抖动
- Y 轴抖动遵循 FMC4030 梯形速度曲线，同一轴前次运动未完成时新指令不响应

In [ ]:
import time
import math
import datetime
from functools import partial
from pathlib import Path
import subprocess

from tqdm.notebook import tqdm, trange
from rich.console import Console
import numpy as np
import ipywidgets as widgets

from mmwave.fmc4030 import FMC4030, Braket
from mmwave.fmc4030.bracket import cal_running_time, cal_acc_time_length
from mmwave.mmwave import MMWaveCmd
from mmwave.util import turn_toml
from mmwave.config import short_range_cfg as mmw_cfg

print = Console().print


def cal_last(use_num, step):
    return step - use_num % step


def cal_all_time(axis_info, step_num):
    all_time = 0
    for axi_info in axis_info:
        axi_time = cal_running_time(*axi_info) * step_num
        all_time += axi_time
    all_time += step_num * 0.1
    return all_time


def gen_jitter_sequence(seed, x_scan_time, y_speed, y_acc, y_dec, amp_min=2.0, amp_max=5.0, cmd_pause=0.025):
    """预生成一行的 Y 轴抖动序列。

    模拟 FMC4030 双轴并行：每次 jog_single_axis_relative 一段梯形速度曲线，
    wait_axis_stop 等待完成，两条指令间有 ~25ms 命令处理延迟。
    返回交替方向的位移列表。
    """
    rng = np.random.default_rng(seed)
    amps = []
    t_budget = x_scan_time - 0.2
    t_used = 0
    direction = 1

    while t_used < t_budget:
        amp = rng.uniform(amp_min, amp_max) * direction
        t_move = cal_running_time(amp, y_speed, y_acc, y_dec)
        if t_used + t_move + cmd_pause > t_budget:
            break
        amps.append(amp)
        t_used += t_move + cmd_pause
        direction *= -1

    return amps


inslider_style = {"description_width": "initial", "width": "50%"}
inslider_layout = widgets.Layout(width="50%")

IntSlider = partial(widgets.IntSlider, style=inslider_style, layout=inslider_layout)
FloatSlider = partial(widgets.FloatSlider, style=inslider_style, layout=inslider_layout)

In [ ]:
acc = 250
sweep_speed = 50
t_acc, len_acc = cal_acc_time_length(acc, speed=sweep_speed)
print(f"加速时间: {t_acc:.2f}秒, 加速长度: {len_acc:.2f}mm")
config_path = "./config.toml"
timestamps_path = "./timestamps.txt"

offset_x = IntSlider(value=0, min=0, max=970, step=1, description="水平扫描偏移(mm)")
offset_y = IntSlider(value=0, min=0, max=1970, step=1, description="垂直扫描偏移(mm)")

fmc4030 = FMC4030()
with Braket(fmc4030) as braket:
    with braket.break_conrtol():
        time.sleep(2)
    braket.jog_x(0)
    braket.jog_y(0)
    braket.home_axis()


@widgets.interact(offset_x=offset_x, offset_y=offset_y)
def to_offset(offset_x, offset_y):
    with Braket(fmc4030) as braket:
        braket.jog_x(offset_x + len_acc)
        braket.jog_y(offset_y)

In [ ]:
sweep_dx = IntSlider(value=1, min=0.6, max=4, step=0.1, description="水平扫描间距(mm)")
sample_points = IntSlider(value=401, min=1, max=901, step=1, description="水平采样点数")

sweep_lines = IntSlider(value=4, min=1, max=20, step=1, description="垂直扫描线数")
sweep_dy = IntSlider(value=80, min=1, max=200, step=1, description="垂直扫描间距(mm)")

# Y 轴抖动参数（模拟手持颤动）
y_jitter_speed = FloatSlider(value=50.0, min=10, max=150, step=5, description="Y抖动速度(mm/s)")
y_jitter_amp_min = FloatSlider(value=2.0, min=0.5, max=10, step=0.5, description="Y抖动最小幅度(mm)")
y_jitter_amp_max = FloatSlider(value=5.0, min=1.0, max=15, step=0.5, description="Y抖动最大幅度(mm)")

Y_JITTER_ACC = 200.0  # mm/s²（与 bracket.py 默认 y_acc 一致）
Y_JITTER_DEC = 200.0


@widgets.interact(
    sweep_dx=sweep_dx,
    sample_points=sample_points,
    sweep_lines=sweep_lines,
    sweep_dy=sweep_dy,
    y_jitter_speed=y_jitter_speed,
    y_jitter_amp_min=y_jitter_amp_min,
    y_jitter_amp_max=y_jitter_amp_max,
)
def set_config(sweep_dx, sample_points, sweep_lines, sweep_dy, y_jitter_speed, y_jitter_amp_min, y_jitter_amp_max):
    framePeriodicity = 1000 * sweep_dx / sweep_speed  # ms
    sweep_length = sweep_dx * (sample_points - 1) + 2 * len_acc
    x_scan_time = cal_running_time(sweep_length, sweep_speed, acc, acc)

    # 预估每行 Y 抖动次数
    avg_amp = (y_jitter_amp_min + y_jitter_amp_max) / 2
    avg_move_time = cal_running_time(avg_amp, y_jitter_speed, Y_JITTER_ACC, Y_JITTER_DEC)
    est_moves = int(x_scan_time / (avg_move_time + 0.025))

    print(f"framePeriodicity: {framePeriodicity} ms")
    print(f"X 扫描长度: {sweep_length:.1f} mm, 扫描时间: {x_scan_time:.2f} s")
    print(f"预估每行 Y 抖动次数: {est_moves}, 单次抖动时间: {avg_move_time:.3f} s")

    axis_info = [
        (sweep_length, sweep_speed, acc, acc),
        (sweep_dy, 150, 200, 200),
        (sweep_length, 200, 250, 250),
    ]
    record_time = cal_all_time(axis_info, sweep_lines) + sweep_lines * 0.5 + 10
    num_frames = int(record_time * 1000 / framePeriodicity)
    print(f"record_time: {record_time:.1f} s, num_frames: {num_frames}")

    mmw_cfg.mimo.frame.framePeriodicity = framePeriodicity
    mmw_cfg.mimo.frame.numLoops = 2
    mmw_cfg.mimo.frame.numFrames = num_frames

    mmw_cfg.bracket.profile.dx = sweep_dx
    mmw_cfg.bracket.profile.dy = sweep_dy
    mmw_cfg.bracket.profile.row = sweep_lines
    mmw_cfg.bracket.profile.col = sample_points
    mmw_cfg.bracket.profile.timestamps = "timestamps.txt"
    mmw_cfg.bracket.profile.offset_time = -0.920
    mmw_cfg.bracket.profile.record_time = record_time
    mmw_cfg.bracket.profile.next_line_reverse = False
    mmw_cfg.bracket.profile.pre_acc = True

In [ ]:
@widgets.interact_manual()
def sweep_handheld():
    now_date = datetime.datetime.now().strftime("%Y%m%d_%H%M")[2:]
    print(now_date)
    mmwave_dir = f"mmw_{now_date}"

    mmw_cfg.bracket.profile.run_time = now_date
    turn_toml(config_path, mmw_cfg.model_dump(exclude_none=True))

    mmwave = MMWaveCmd(config_file=config_path)

    record_time = mmw_cfg.bracket.profile.record_time
    framePeriodicity = mmw_cfg.mimo.frame.framePeriodicity
    sweep_length = sweep_dx.value * (sample_points.value - 1) + 2 * len_acc
    x_scan_time = cal_running_time(sweep_length, sweep_speed, acc, acc)

    # ── 预生成所有行的 Y 抖动序列 ──
    jitter_seqs = [
        gen_jitter_sequence(
            seed=42 + line * 7,
            x_scan_time=x_scan_time,
            y_speed=y_jitter_speed.value,
            y_acc=Y_JITTER_ACC,
            y_dec=Y_JITTER_DEC,
            amp_min=y_jitter_amp_min.value,
            amp_max=y_jitter_amp_max.value,
        )
        for line in range(sweep_lines.value)
    ]
    for i, seq in enumerate(jitter_seqs):
        print(f"  行 {i}: {len(seq)} 次抖动, 净偏移 {sum(seq):+.2f} mm")

    time_list = []
    jitter_records = []  # 记录每行实际执行的抖动
    jog_x_delay = 0

    with mmwave.record(mmwave_dir, record_time):
        st = time.time()
        with Braket(fmc4030) as braket:
            braket.jog_x(offset_x.value)

            for y_line in trange(sweep_lines.value):
                # 帧同步等待
                time.sleep(
                    cal_last(
                        time.time() - st + t_acc + jog_x_delay,
                        framePeriodicity / 1000,
                    )
                )
                start_time = time.time()

                # ══════════════════════════════════════
                # 双轴并行：X 匀速扫描 + Y 梯形抖动
                # ══════════════════════════════════════

                # 1) 解锁 Y 轴抱闸（Y 轴映射到控制器 Z 轴，抱闸 IO=0）
                braket.bc.set_output(0, 1)

                # 2) 启动 X 扫描（非阻塞——不调用 wait_axis_stop）
                x_target = sweep_length + offset_x.value
                real_x = braket._real_pos(x_target, braket.x_reverse)
                braket.bc.jog_single_axis_absolute(
                    braket.x_axis_id,
                    real_x,
                    sweep_speed,
                    acc,
                    acc,
                )

                # 3) X 运动期间循环执行 Y 抖动
                #    控制逻辑：jog_single_axis_relative → wait_axis_stop → 下一条
                #    同一轴前次运动未完成时新指令不响应
                y_cumulative = 0.0
                executed_amps = []
                for amp in jitter_seqs[y_line]:
                    if braket.bc.check_axis_is_stop(braket.x_axis_id):
                        break
                    real_y_rel = braket._real_pos(amp, braket.y_reverse)
                    braket.bc.jog_single_axis_relative(
                        braket.y_axis_id,
                        real_y_rel,
                        y_jitter_speed.value,
                        Y_JITTER_ACC,
                        Y_JITTER_DEC,
                    )
                    braket.bc.wait_axis_stop(braket.y_axis_id)
                    y_cumulative += amp
                    executed_amps.append(amp)

                # 4) 等待 X 扫描完成
                braket.bc.wait_axis_stop(braket.x_axis_id)
                braket.x_pos = x_target

                record_end = time.time() - st - t_acc + jog_x_delay

                # 5) 锁定 Y 轴抱闸
                braket.bc.set_output(0, 0)

                # 记录本行抖动数据
                jitter_records.append(executed_amps)
                tqdm.write(
                    f"行 {y_line}: 执行 {len(executed_amps)}/{len(jitter_seqs[y_line])} 次抖动, 净偏移 {y_cumulative:+.2f} mm"
                )

                # ══════════════════════════════════════
                # 行间步进：Y 绝对定位到下一行，X 返回起点
                # ══════════════════════════════════════
                braket.y_pos = y_line * sweep_dy.value + offset_y.value + y_cumulative
                braket.jog_y((y_line + 1) * sweep_dy.value + offset_y.value)
                braket.jog_x(0 + offset_x.value)

                record_start = start_time - st + t_acc + jog_x_delay
                tqdm.write(f"  时间窗: ({record_start:.3f}, {record_end:.3f})")
                time_list.append((record_start, record_end))

            braket.jog_x(0)
            braket.jog_y(0)

        time_offset = mmwave.sync_time(st)
        time_list.append([time_offset, 0])
        np.savetxt(timestamps_path, np.array(time_list))

        # 保存抖动轨迹数据供后续分析
        jitter_save_path = f"./{mmwave_dir}_jitter.npz"
        np.savez(
            jitter_save_path,
            **{f"line_{i}": np.array(amps) for i, amps in enumerate(jitter_records)},
        )
        print(f"抖动数据已保存: {jitter_save_path}")

        wait_time = record_time - (time.time() - st)
        time.sleep(max(0, wait_time))

In [ ]:
from mmwave.util import load_config


def get_dev_files(dir_path: str):
    res = subprocess.run(["ssh", "root@192.168.33.180", f"ls /mnt/ssd/{dir_path}"], capture_output=True)
    res = res.stdout.decode()
    files = [i for i in res.split()]
    return files


@widgets.interact_manual()
def copy_config():
    mmw_cfg_local = load_config(config_path)
    now_date = mmw_cfg_local.bracket.profile.run_time
    mmwave_dir = f"mmw_{now_date}"

    files = get_dev_files(mmwave_dir)
    if config_path[2:] not in files:
        subprocess.run(["rsync", "-av", config_path, f"root@192.168.33.180:/mnt/ssd/{mmwave_dir}/"])
    if timestamps_path[2:] not in files:
        subprocess.run(["rsync", "-av", timestamps_path, f"root@192.168.33.180:/mnt/ssd/{mmwave_dir}/"])

In [ ]:
@widgets.interact_manual()
def ls_file():
    now_date = mmw_cfg.bracket.profile.run_time
    mmwave_dir = f"mmw_{now_date}"

    files = get_dev_files(mmwave_dir)
    print(files)

In [ ]:
@widgets.interact_manual()
def download_data():
    mmw_cfg_local = load_config(config_path)
    now_date = mmw_cfg_local.bracket.profile.run_time
    mmwave_dir = f"mmw_{now_date}"

    # kill apps.out
    cmd = ["ssh", "root@192.168.33.180", "-C", "killall apps.out"]
    subprocess.run(cmd)

    tartgetdir_path = f"/media/ubuntu/tomin/{mmwave_dir}"
    Path(tartgetdir_path).mkdir(exist_ok=True)
    for f in tqdm(get_dev_files(mmwave_dir)):
        cmd = [
            "rsync",
            "-av",
            "--progress",
            "--whole-file",
            f"root@192.168.33.180:/mnt/ssd/{mmwave_dir}/{f}",
            f"{tartgetdir_path}/{f}",
        ]
        subprocess.run(cmd)